# Khám phá bộ dữ liệu CICIDS2017

Notebook này khảo sát CICIDS2017 ở dạng CSV, trong đó mỗi dòng mô tả một network flow cùng các đặc trưng thống kê và nhãn. Mục tiêu đồ án là phân loại nhị phân `Normal` hoặc `Attack`; các nhãn gốc như `DDoS` được dùng để tìm hiểu dữ liệu và đánh giá chi tiết, không phải đầu ra dự đoán cuối cùng.

Các mục 1–14 khảo sát cấu trúc, nhãn và chất lượng dữ liệu trên 8 CSV; mục 15 tổng kết Data Exploration và nêu nguyên tắc chuyển sang Data Preprocessing. Các con số trong notebook là kết quả của những cell đã chạy. **Chưa làm sạch dữ liệu, chia Train/Test hoặc huấn luyện mô hình**. Bài báo GVHD cung cấp được ghi chú riêng ở `../docs/paper_notes.md`; định hướng thử Random Split và Day-Disjoint về sau không thay đổi mục đích Data Exploration hiện tại.

## 1. Đọc dữ liệu và kiểm tra cấu trúc

Cell này nạp file `Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv` từ thư mục dataset và loại khoảng trắng thừa ở tên cột. Đây là một file trong CICIDS2017, chưa phải toàn bộ dữ liệu.

In [17]:
import pandas as pd

df = pd.read_csv(
    "../Dataset/CIC-IDS-2017/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
)

df.columns = df.columns.str.strip()

### 1.1. Kích thước dữ liệu

Kích thước cho biết số flow (dòng) và số trường (cột) của file đang khảo sát. Output hiện lưu là `(225745, 79)`.

In [18]:
df.shape

(225745, 79)

### 1.2. Quan sát một số bản ghi

Xem vài dòng đầu giúp nhận biết cách dữ liệu được biểu diễn: mỗi dòng có các đặc trưng flow và nhãn. Đây là bước kiểm tra trực quan, không đại diện cho toàn bộ phân bố dữ liệu.

In [19]:
df.head()


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


### 1.3. Danh sách cột

Liệt kê tên cột để biết những đặc trưng và trường nhãn có trong file. File này có 78 cột đặc trưng cùng cột `Label`.

In [20]:
df.columns.tolist()

['Destination Port',
 'Flow Duration',
 'Total Fwd Packets',
 'Total Backward Packets',
 'Total Length of Fwd Packets',
 'Total Length of Bwd Packets',
 'Fwd Packet Length Max',
 'Fwd Packet Length Min',
 'Fwd Packet Length Mean',
 'Fwd Packet Length Std',
 'Bwd Packet Length Max',
 'Bwd Packet Length Min',
 'Bwd Packet Length Mean',
 'Bwd Packet Length Std',
 'Flow Bytes/s',
 'Flow Packets/s',
 'Flow IAT Mean',
 'Flow IAT Std',
 'Flow IAT Max',
 'Flow IAT Min',
 'Fwd IAT Total',
 'Fwd IAT Mean',
 'Fwd IAT Std',
 'Fwd IAT Max',
 'Fwd IAT Min',
 'Bwd IAT Total',
 'Bwd IAT Mean',
 'Bwd IAT Std',
 'Bwd IAT Max',
 'Bwd IAT Min',
 'Fwd PSH Flags',
 'Bwd PSH Flags',
 'Fwd URG Flags',
 'Bwd URG Flags',
 'Fwd Header Length',
 'Bwd Header Length',
 'Fwd Packets/s',
 'Bwd Packets/s',
 'Min Packet Length',
 'Max Packet Length',
 'Packet Length Mean',
 'Packet Length Std',
 'Packet Length Variance',
 'FIN Flag Count',
 'SYN Flag Count',
 'RST Flag Count',
 'PSH Flag Count',
 'ACK Flag Count',
 'UR

## 2. Khảo sát nhãn trong file DDoS

Giá trị duy nhất của `Label` cho biết các lớp gốc xuất hiện trong file đang nạp. Output hiện lưu gồm `BENIGN` và `DDoS`.

In [21]:
df["Label"].unique()


array(['BENIGN', 'DDoS'], dtype=object)

### 2.1. Số lượng mẫu theo nhãn

Đếm từng nhãn để thấy quy mô lớp bình thường và lớp tấn công trong file. Đây là số liệu của file DDoS, chưa phải thống kê gộp toàn bộ CICIDS2017.

In [22]:
df["Label"].value_counts()


Label
DDoS      128027
BENIGN     97718
Name: count, dtype: int64

## 3. Kiểm tra chất lượng dữ liệu

NaN là giá trị thiếu; Infinity là giá trị vô hạn, thường cần chú ý ở các đặc trưng tốc độ. Trước khi tiền xử lý, ta cần xác định chúng có xuất hiện hay không và nằm ở đâu. Output tổng hiện lưu ghi nhận 4 NaN và 64 Infinity trong file đang khảo sát.

In [23]:
import numpy as np

print("NaN count:")
print(df.isna().sum().sum())

print("Infinity count:")
print(np.isinf(df.select_dtypes(include=np.number)).sum().sum())

NaN count:
4
Infinity count:
64


### 3.1. Xác định cột có NaN hoặc Infinity

Thống kê theo từng cột giúp xác định vấn đề tập trung ở đặc trưng nào. Output hiện lưu: `Flow Bytes/s` có 4 NaN và 30 Infinity; `Flow Packets/s` có 34 Infinity.

In [24]:
# Số lượng NaN theo từng cột
nan_counts = df.isna().sum()
print(nan_counts[nan_counts > 0])

# Số lượng Infinity theo từng cột số
numeric_df = df.select_dtypes(include=np.number)

inf_counts = np.isinf(numeric_df).sum()
print(inf_counts[inf_counts > 0])

Flow Bytes/s    4
dtype: int64
Flow Bytes/s      30
Flow Packets/s    34
dtype: int64


### 3.2. Xem các dòng có giá trị không hợp lệ

Lọc dòng có NaN hoặc Infinity để quan sát ngữ cảnh. Các dòng hiển thị có `Flow Duration = 0`; đây là quan sát cần điều tra, chưa phải kết luận nguyên nhân và chưa áp dụng quy tắc xóa/thay thế.

In [25]:
invalid_rows = df[
    df.isna().any(axis=1) |
    np.isinf(df.select_dtypes(include=np.number)).any(axis=1)
]

invalid_rows

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
65,55043,0,2,0,12,0,6,6,6.0,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1767,60018,0,2,0,12,0,6,6,6.0,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1890,80,0,1,1,6,6,6,6,6.0,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3375,443,0,1,1,6,6,6,6,6.0,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
6796,36812,0,2,0,0,0,0,0,0.0,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
8057,35382,0,1,1,6,0,6,6,6.0,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
8405,55421,0,2,0,20,0,14,6,10.0,5.656854,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
13313,9242,0,2,0,12,0,6,6,6.0,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
13716,389,0,2,0,7,0,7,0,3.5,4.949747,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
14739,53581,0,1,1,0,0,0,0,0.0,0.000000,...,44,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


### 3.3. Kiểm tra dòng trùng

`duplicated()` đếm những dòng có toàn bộ giá trị trùng với **một dòng đã xuất hiện trước đó trong DataFrame**, không nhất thiết là dòng đứng ngay trước. Output ghi nhận 2.633 dòng trùng trong riêng file DDoS; chưa loại bỏ vì cần phân tích ý nghĩa và nguy cơ ảnh hưởng đến cách chia Train/Test.

In [26]:
print("Duplicated rows:", df.duplicated().sum())

Duplicated rows: 2633


## 4. Thống kê mô tả đặc trưng

`describe()` tóm tắt các thống kê như trung bình, độ lệch chuẩn, min, các tứ phân vị và max của cột số. Output có warning liên quan phép tính khi dữ liệu chứa Infinity, nên một số thống kê cần được xem xét sau khi xử lý dữ liệu. Output cũng cho thấy `Flow Duration` có min bằng -1; số lượng và cách xử lý giá trị này chưa được xác định.

In [27]:
df.describe()

D:\HUST\GR2\NIDS-ML\.venv\lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
D:\HUST\GR2\NIDS-ML\.venv\lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
count,225745.00000,2.257450e+05,225745.000000,225745.000000,225745.000000,2.257450e+05,225745.000000,225745.000000,225745.000000,225745.000000,...,225745.000000,225745.000000,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05
mean,8879.61946,1.624165e+07,4.874916,4.572775,939.463346,5.960477e+03,538.535693,27.882221,164.826715,214.907242,...,3.311497,21.482753,1.848261e+05,1.293436e+04,2.080849e+05,1.776201e+05,1.032214e+07,3.611943e+06,1.287813e+07,7.755355e+06
std,19754.64740,3.152437e+07,15.422874,21.755356,3249.403484,3.921834e+04,1864.128991,163.324159,504.892965,797.411073,...,12.270018,4.166799,7.979250e+05,2.102737e+05,9.002350e+05,7.842602e+05,2.185303e+07,1.275689e+07,2.692126e+07,1.983109e+07
min,0.00000,-1.000000e+00,1.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,80.00000,7.118000e+04,2.000000,1.000000,26.000000,0.000000e+00,6.000000,0.000000,6.000000,0.000000,...,1.000000,20.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,80.00000,1.452333e+06,3.000000,4.000000,30.000000,1.640000e+02,20.000000,0.000000,8.666667,5.301991,...,2.000000,20.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,80.00000,8.805237e+06,5.000000,5.000000,63.000000,1.160100e+04,34.000000,6.000000,32.000000,10.263203,...,4.000000,20.000000,1.878000e+03,0.000000e+00,1.878000e+03,1.862000e+03,8.239725e+06,0.000000e+00,8.253838e+06,7.422849e+06
max,65532.00000,1.199999e+08,1932.000000,2942.000000,183012.000000,5.172346e+06,11680.000000,1472.000000,3867.000000,6692.644993,...,1931.000000,52.000000,1.000000e+08,3.950000e+07,1.000000e+08,1.000000e+08,1.200000e+08,6.530000e+07,1.200000e+08,1.200000e+08


## 5. Tìm đặc trưng không đổi

Cột có một giá trị duy nhất trong file này không giúp phân biệt các dòng của chính file đó. Output liệt kê 10 cột hằng số. Chưa kết luận có thể xóa chúng khỏi toàn bộ dataset: cần kiểm tra trên phạm vi dữ liệu phù hợp trước.

In [28]:
constant_columns = [
    col for col in df.columns
    if df[col].nunique(dropna=False) <= 1
]

constant_columns

['Bwd PSH Flags',
 'Fwd URG Flags',
 'Bwd URG Flags',
 'CWE Flag Count',
 'Fwd Avg Bytes/Bulk',
 'Fwd Avg Packets/Bulk',
 'Fwd Avg Bulk Rate',
 'Bwd Avg Bytes/Bulk',
 'Bwd Avg Packets/Bulk',
 'Bwd Avg Bulk Rate']

## 6. Khảo sát nhãn trên 8 file CSV

Vòng lặp đọc từng file và in số lượng theo `Label` để xem các nhãn gốc và quy mô từng file. Các output hiện lưu có kết quả cho 8 CSV. Đây chỉ là thống kê nhãn; cell này chưa tổng hợp NaN, Infinity, duplicate hay ghép dữ liệu.

In [29]:
from pathlib import Path
import pandas as pd

data_dir = Path("../Dataset/CIC-IDS-2017")

for file in data_dir.glob("*.csv"):
    temp = pd.read_csv(file)
    temp.columns = temp.columns.str.strip()

    print("\nFILE:", file.name)
    print(temp["Label"].value_counts())


FILE: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Label
DDoS      128027
BENIGN     97718
Name: count, dtype: int64

FILE: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Label
PortScan    158930
BENIGN      127537
Name: count, dtype: int64

FILE: Friday-WorkingHours-Morning.pcap_ISCX.csv
Label
BENIGN    189067
Bot         1966
Name: count, dtype: int64

FILE: Monday-WorkingHours.pcap_ISCX.csv
Label
BENIGN    529918
Name: count, dtype: int64

FILE: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Label
BENIGN          288566
Infiltration        36
Name: count, dtype: int64

FILE: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Label
BENIGN                        168186
Web Attack � Brute Force        1507
Web Attack � XSS                 652
Web Attack � Sql Injection        21
Name: count, dtype: int64

FILE: Tuesday-WorkingHours.pcap_ISCX.csv
Label
BENIGN         432074
FTP-Patator      7938
SSH-Patator      5897
Name: count, dtype: int64

FILE: Wednesday

## 7. Ánh xạ nhãn sang bài toán nhị phân

Cell này thêm `BinaryLabel` vào `df` hiện tại: `BENIGN` được ánh xạ thành `Normal`, còn nhãn khác thành `Attack`. Vì `df` hiện là file DDoS nên kết quả ở đây chỉ xác nhận ánh xạ trên file đó: 97.718 Normal và 128.027 Attack. Chưa áp dụng ánh xạ hoặc tạo bộ dữ liệu sạch cho toàn bộ 8 file.

In [30]:
df["BinaryLabel"] = df["Label"].apply(
    lambda x: "Normal" if x == "BENIGN" else "Attack"
)

df["BinaryLabel"].value_counts()

BinaryLabel
Attack    128027
Normal     97718
Name: count, dtype: int64

## 8. Khảo sát chất lượng dữ liệu trên 8 file CSV

Các mục 3–5 mới kiểm tra file DDoS. Cell dưới đây đọc lần lượt từng CSV và thống kê: số dòng, số cột, số ô `NaN`, số ô `Infinity` (ở các cột số) và số dòng trùng hoàn toàn **trong nội bộ từng file**.

Lưu ý:
- File được đọc bằng `encoding="latin-1"` vì nhãn `Web Attack` chứa ký tự không phải UTF-8.
- Tổng `Duplicates` chỉ cộng trùng nội bộ từng file; dòng trùng giữa các file khác nhau chưa được tính.
- Cell chỉ thống kê, không xóa hay sửa dữ liệu.

In [31]:
quality_rows = []

for file in sorted(data_dir.glob("*.csv")):
    temp = pd.read_csv(file, encoding="latin-1")
    temp.columns = temp.columns.str.strip()

    numeric_part = temp.select_dtypes(include=np.number)

    quality_rows.append({
        "File": file.name,
        "Rows": temp.shape[0],
        "Columns": temp.shape[1],
        "NaN": int(temp.isna().sum().sum()),
        "Infinity": int(np.isinf(numeric_part).sum().sum()),
        "Duplicates": int(temp.duplicated().sum()),
    })

    del temp, numeric_part

quality_df = pd.DataFrame(quality_rows)
total_row = quality_df.drop(columns="File").sum()
quality_df.loc[len(quality_df)] = ["TOTAL", *total_row]
quality_df

,File,Rows,Columns,NaN,Infinity,Duplicates
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,79,4,64,2633
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,79,15,727,72353
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,79,28,216,6888
3,Monday-WorkingHours.pcap_ISCX.csv,529918,79,64,810,26935
4,Thursday-WorkingHours-Afternoon-Infilteration....,288602,79,18,396,35630
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170366,79,20,250,6066
6,Tuesday-WorkingHours.pcap_ISCX.csv,445909,79,201,327,24065
7,Wednesday-workingHours.pcap_ISCX.csv,692703,79,1008,1586,81909
8,TOTAL,2830743,632,1358,4376,256479


## 9. Xác định vị trí các giá trị cần điều tra

Mục 8 cho biết mỗi file có bao nhiêu giá trị thiếu (`NaN`), vô hạn (`Infinity`) và dòng trùng, nhưng chưa cho biết chúng nằm ở cột nào hay thuộc lớp nào. Cell tiếp theo đọc từng file để thống kê số dòng bị ảnh hưởng theo nhãn nhị phân `Normal`/`Attack`, kiểm tra các dòng lỗi có `Flow Duration = 0` hay không, đồng thời đếm giá trị âm hữu hạn theo từng cột số.

Các bảng chỉ mô tả dữ liệu, không sửa hoặc loại bỏ dòng. Giá trị âm được báo cáo để điều tra, chưa mặc định là lỗi; `-Infinity` cũng không được tính là giá trị âm hữu hạn.

In [32]:
issue_by_label_rows = []
issue_by_column_rows = []

for file in sorted(data_dir.glob("*.csv")):
    temp = pd.read_csv(file, encoding="latin-1")
    temp.columns = temp.columns.str.strip()
    numeric_part = temp.select_dtypes(include=np.number)

    nan_cells_by_column = temp.isna().sum()
    inf_cells_by_column = np.isinf(numeric_part).sum()
    finite_negative_by_column = (
        numeric_part.lt(0) & np.isfinite(numeric_part)
    ).sum()

    affected_columns = set(nan_cells_by_column[nan_cells_by_column > 0].index)
    affected_columns.update(inf_cells_by_column[inf_cells_by_column > 0].index)
    affected_columns.update(
        finite_negative_by_column[finite_negative_by_column > 0].index
    )

    for column in sorted(affected_columns):
        issue_by_column_rows.append({
            "File": file.name,
            "Column": column,
            "NaN cells": int(nan_cells_by_column.get(column, 0)),
            "Infinity cells": int(inf_cells_by_column.get(column, 0)),
            "Finite negative values": int(finite_negative_by_column.get(column, 0)),
        })

    nan_row_mask = temp.isna().any(axis=1)
    inf_row_mask = np.isinf(numeric_part).any(axis=1)
    invalid_row_mask = nan_row_mask | inf_row_mask
    duration_zero_mask = temp["Flow Duration"].eq(0)
    binary_label = temp["Label"].eq("BENIGN").map({True: "Normal", False: "Attack"})

    for label in ("Normal", "Attack"):
        label_mask = binary_label.eq(label)
        issue_by_label_rows.append({
            "File": file.name,
            "BinaryLabel": label,
            "Rows with NaN": int((nan_row_mask & label_mask).sum()),
            "Rows with Infinity": int((inf_row_mask & label_mask).sum()),
            "Rows with NaN or Infinity": int((invalid_row_mask & label_mask).sum()),
            "Invalid rows with Flow Duration = 0": int(
                (invalid_row_mask & duration_zero_mask & label_mask).sum()
            ),
        })

    del temp, numeric_part

issue_by_label = pd.DataFrame(issue_by_label_rows)
issue_by_column = pd.DataFrame(issue_by_column_rows)

print("Số dòng có vấn đề theo file và nhãn:")
display(issue_by_label)

print("Các cột có NaN, Infinity hoặc giá trị âm hữu hạn:")
display(issue_by_column)

Số dòng có vấn đề theo file và nhãn:


,File,BinaryLabel,Rows with NaN,Rows with Infinity,Rows with NaN or Infinity,Invalid rows with Flow Duration = 0
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Normal,4,32,32,32
1,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Attack,0,2,2,2
2,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Normal,15,245,245,245
3,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Attack,0,126,126,126
4,Friday-WorkingHours-Morning.pcap_ISCX.csv,Normal,28,112,112,112
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,Attack,0,10,10,10
6,Monday-WorkingHours.pcap_ISCX.csv,Normal,64,437,437,437
7,Monday-WorkingHours.pcap_ISCX.csv,Attack,0,0,0,0
8,Thursday-WorkingHours-Afternoon-Infilteration....,Normal,18,207,207,207
9,Thursday-WorkingHours-Afternoon-Infilteration....,Attack,0,0,0,0


Các cột có NaN, Infinity hoặc giá trị âm hữu hạn:


,File,Column,NaN cells,Infinity cells,Finite negative values
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Flow Bytes/s,4,30,2
1,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Flow Duration,0,0,2
2,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Flow IAT Max,0,0,2
3,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Flow IAT Mean,0,0,2
4,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Flow IAT Min,0,0,108
...,...,...,...,...,...
72,Wednesday-workingHours.pcap_ISCX.csv,Fwd Header Length.1,0,0,1
73,Wednesday-workingHours.pcap_ISCX.csv,Fwd IAT Min,0,0,11
74,Wednesday-workingHours.pcap_ISCX.csv,Init_Win_bytes_backward,0,0,335475
75,Wednesday-workingHours.pcap_ISCX.csv,Init_Win_bytes_forward,0,0,203253


## 10. Khảo sát giá trị âm hữu hạn

Mục 9 cho thấy phần lớn giá trị âm hữu hạn nằm ở `Init_Win_bytes_forward` và `Init_Win_bytes_backward`, nhưng chưa biết đó là những trị số nào. Cell dưới đây đọc lại 8 CSV (`encoding="latin-1"`) và chỉ thống kê, không sửa hoặc loại bỏ dữ liệu:

1. `negative_summary`: với mỗi cột có số âm, cho biết số ô âm, số trị số âm khác nhau, giá trị nhỏ nhất, số ô bằng `-1` và số ô âm khác `-1`.
2. `negative_top_values`: các trị số âm xuất hiện nhiều nhất của từng cột.
3. `negative_by_label`: số ô âm theo cột và nhãn `Normal`/`Attack`.

Mục đích là kiểm tra giả thuyết `-1` có thể là giá trị quy ước, còn các số âm khác cần xem riêng. Đây mới là giả thuyết, chưa phải kết luận.

In [33]:
from collections import Counter

negative_counter = {}
negative_label_counter = Counter()

for file in sorted(data_dir.glob("*.csv")):
    temp = pd.read_csv(file, encoding="latin-1")
    temp.columns = temp.columns.str.strip()
    numeric_part = temp.select_dtypes(include=np.number)
    binary_label = temp["Label"].eq("BENIGN").map({True: "Normal", False: "Attack"})

    for column in numeric_part.columns:
        values = numeric_part[column]
        negative_mask = values.lt(0) & np.isfinite(values)
        if not negative_mask.any():
            continue

        negative_counter.setdefault(column, Counter()).update(
            values[negative_mask].value_counts().to_dict()
        )
        for label, count in binary_label[negative_mask].value_counts().items():
            negative_label_counter[(column, label)] += int(count)

    del temp, numeric_part

summary_rows = []
top_rows = []

for column, counter in negative_counter.items():
    total = sum(counter.values())
    summary_rows.append({
        "Column": column,
        "Negative cells": total,
        "Distinct negative values": len(counter),
        "Minimum": min(counter),
        "Cells equal to -1": counter.get(-1, 0),
        "Other negative cells": total - counter.get(-1, 0),
    })
    for value, count in counter.most_common(5):
        top_rows.append({"Column": column, "Value": value, "Cells": count})

negative_summary = (
    pd.DataFrame(summary_rows)
    .sort_values("Negative cells", ascending=False)
    .reset_index(drop=True)
)
negative_top_values = pd.DataFrame(top_rows)
negative_by_label = (
    pd.DataFrame(
        [(c, l, n) for (c, l), n in negative_label_counter.items()],
        columns=["Column", "BinaryLabel", "Negative cells"],
    )
    .pivot(index="Column", columns="BinaryLabel", values="Negative cells")
    .fillna(0)
    .astype(int)
    .reset_index()
)

pd.set_option("display.max_rows", None)

print("Tổng hợp giá trị âm hữu hạn theo cột:")
display(negative_summary)

print("Các trị số âm xuất hiện nhiều nhất:")
display(negative_top_values)

print("Số ô âm theo cột và nhãn nhị phân:")
display(negative_by_label)

Tổng hợp giá trị âm hữu hạn theo cột:


,Column,Negative cells,Distinct negative values,Minimum,Cells equal to -1,Other negative cells
0,Init_Win_bytes_backward,1441552,1,-1.000000e+00,1441552,0
1,Init_Win_bytes_forward,1001189,1,-1.000000e+00,1001189,0
2,Flow IAT Min,2891,12,-1.400000e+01,2754,137
3,Flow Packets/s,115,5,-2.000000e+06,0,115
4,Flow Duration,115,5,-1.300000e+01,109,6
5,Flow IAT Mean,115,5,-1.300000e+01,109,6
6,Flow IAT Max,115,5,-1.300000e+01,109,6
7,Flow Bytes/s,85,9,-2.610000e+08,0,85
8,Fwd Header Length,35,27,-3.221223e+10,0,35
9,min_seg_size_forward,35,4,-5.368707e+08,1,34


Các trị số âm xuất hiện nhiều nhất:


,Column,Value,Cells
0,Flow Duration,-1.000000e+00,109
1,Flow Duration,-1.200000e+01,2
2,Flow Duration,-2.000000e+00,2
3,Flow Duration,-1.300000e+01,1
4,Flow Duration,-4.000000e+00,1
5,Flow Bytes/s,-1.200000e+07,50
6,Flow Bytes/s,-6.000000e+06,16
7,Flow Bytes/s,-8.000000e+06,13
8,Flow Bytes/s,-1.930000e+08,1
9,Flow Bytes/s,-6.666667e+05,1


Số ô âm theo cột và nhãn nhị phân:


BinaryLabel,Column,Attack,Normal
0,Bwd Header Length,0,22
1,Flow Bytes/s,0,85
2,Flow Duration,0,115
3,Flow IAT Max,0,115
4,Flow IAT Mean,0,115
5,Flow IAT Min,194,2697
6,Flow Packets/s,0,115
7,Fwd Header Length,0,35
8,Fwd Header Length.1,0,35
9,Fwd IAT Min,17,0


## 11. Đối chiếu các dòng có giá trị âm khác `-1`

Mục này kiểm tra theo dòng, không chỉ đếm số ô. Cell tập trung vào các giá trị âm hữu hạn khác `-1`, đồng thời đối chiếu với `Flow Duration`, `Flow Bytes/s`, `Flow Packets/s`, NaN/Infinity, nhãn gốc và file nguồn.

Các bảng được tạo: `negative_row_summary` theo file/nhãn; `negative_by_column_label` theo cột/file/nhãn; và `negative_examples` gồm tối đa 20 dòng đầu tiên của mỗi file. Cell chỉ mô tả dữ liệu; không sửa, thay thế hoặc loại bỏ dòng.


In [34]:
negative_row_summary_rows = []
negative_column_label_rows = []
negative_example_frames = []

for file in sorted(data_dir.glob("*.csv")):
    temp = pd.read_csv(file, encoding="latin-1")
    temp.columns = temp.columns.str.strip()
    numeric_part = temp.select_dtypes(include=np.number)
    finite_negative = numeric_part.lt(0) & np.isfinite(numeric_part)
    non_sentinel_negative = finite_negative & numeric_part.ne(-1)
    affected_row_mask = non_sentinel_negative.any(axis=1)
    nan_row_mask = temp.isna().any(axis=1)
    inf_row_mask = np.isinf(numeric_part).any(axis=1)
    binary_label = temp["Label"].eq("BENIGN").map({True: "Normal", False: "Attack"})

    for label in ("Normal", "Attack"):
        rows = affected_row_mask & binary_label.eq(label)
        negative_row_summary_rows.append({
            "File": file.name,
            "BinaryLabel": label,
            "Rows with negative values other than -1": int(rows.sum()),
            "Rows with NaN": int((rows & nan_row_mask).sum()),
            "Rows with Infinity": int((rows & inf_row_mask).sum()),
            "Rows with Flow Duration = 0": int((rows & temp["Flow Duration"].eq(0)).sum()),
        })

    for column in numeric_part.columns:
        column_mask = non_sentinel_negative[column]
        if column_mask.any():
            counts = binary_label[column_mask].value_counts()
            for label in ("Normal", "Attack"):
                negative_column_label_rows.append({"File": file.name, "Column": column, "BinaryLabel": label, "Rows": int(counts.get(label, 0))})

    if affected_row_mask.any():
        related_columns = ["Label", "Flow Duration", "Flow Bytes/s", "Flow Packets/s", "Flow IAT Min", "Flow IAT Mean", "Flow IAT Max", "Fwd IAT Min", "Init_Win_bytes_forward", "Init_Win_bytes_backward", "Fwd Header Length", "Fwd Header Length.1", "Bwd Header Length", "min_seg_size_forward"]
        example = temp.loc[affected_row_mask, [c for c in related_columns if c in temp.columns]].head(20).copy()
        example.insert(0, "SourceFile", file.name)
        example.insert(1, "BinaryLabel", binary_label.loc[example.index].values)
        negative_example_frames.append(example.reset_index(names="SourceRow"))
    del temp, numeric_part

negative_row_summary = pd.DataFrame(negative_row_summary_rows)
negative_by_column_label = pd.DataFrame(negative_column_label_rows)
negative_examples = pd.concat(negative_example_frames, ignore_index=True) if negative_example_frames else pd.DataFrame()
print("Số dòng có âm hữu hạn khác -1 theo file và nhãn:")
display(negative_row_summary)
print("Số dòng có âm hữu hạn khác -1 theo cột, file và nhãn:")
display(negative_by_column_label[negative_by_column_label["Rows"] > 0].sort_values(["Column", "File", "BinaryLabel"]))
print("Mẫu tối đa 20 dòng đầu tiên của mỗi file có âm khác -1:")
display(negative_examples)


Số dòng có âm hữu hạn khác -1 theo file và nhãn:


,File,BinaryLabel,Rows with negative values other than -1,Rows with NaN,Rows with Infinity,Rows with Flow Duration = 0
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Normal,5,0,0,0
1,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Attack,1,0,0,0
2,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Normal,41,0,0,0
3,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Attack,0,0,0,0
4,Friday-WorkingHours-Morning.pcap_ISCX.csv,Normal,24,0,0,0
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,Attack,0,0,0,0
6,Monday-WorkingHours.pcap_ISCX.csv,Normal,61,0,0,0
7,Monday-WorkingHours.pcap_ISCX.csv,Attack,0,0,0,0
8,Thursday-WorkingHours-Afternoon-Infilteration....,Normal,16,0,0,0
9,Thursday-WorkingHours-Afternoon-Infilteration....,Attack,0,0,0,0


Số dòng có âm hữu hạn khác -1 theo cột, file và nhãn:


,File,Column,BinaryLabel,Rows
40,Monday-WorkingHours.pcap_ISCX.csv,Bwd Header Length,Normal,17
76,Tuesday-WorkingHours.pcap_ISCX.csv,Bwd Header Length,Normal,5
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Flow Bytes/s,Normal,2
10,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Flow Bytes/s,Normal,36
22,Friday-WorkingHours-Morning.pcap_ISCX.csv,Flow Bytes/s,Normal,8
32,Monday-WorkingHours.pcap_ISCX.csv,Flow Bytes/s,Normal,9
48,Thursday-WorkingHours-Afternoon-Infilteration....,Flow Bytes/s,Normal,4
58,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,Flow Bytes/s,Normal,10
82,Wednesday-workingHours.pcap_ISCX.csv,Flow Bytes/s,Normal,16
8,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Flow Duration,Normal,3


Mẫu tối đa 20 dòng đầu tiên của mỗi file có âm khác -1:


,SourceRow,SourceFile,BinaryLabel,Label,Flow Duration,Flow Bytes/s,Flow Packets/s,Flow IAT Min,Flow IAT Mean,Flow IAT Max,Fwd IAT Min,Init_Win_bytes_forward,Init_Win_bytes_backward,Fwd Header Length,Fwd Header Length.1,Bwd Header Length,min_seg_size_forward
0,96829,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Attack,DDoS,20933,5.557254e+05,3.344002e+02,-12,3.488833e+03,20325,-12,29200,229,104,104,60,20
1,200991,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Normal,BENIGN,-1,-1.200000e+07,-2.000000e+06,-1,-1.000000e+00,-1,0,63982,30016,20,20,20,20
2,204805,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Normal,BENIGN,76021085,4.758653e+04,3.831832e+01,-3,2.610614e+04,10100000,1,29200,319,36936,36936,56296,32
3,209295,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Normal,BENIGN,117778260,8.361237e+03,7.233933e+00,-2,1.383998e+05,58900000,1,8192,292,7752,7752,9312,20
4,212095,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Normal,BENIGN,63387803,3.161799e+04,2.973758e+01,-2,3.364533e+04,58400000,0,8192,1549,24032,24032,19032,20
5,224100,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Normal,BENIGN,-1,-1.200000e+07,-2.000000e+06,-1,-1.000000e+00,-1,0,16425,946,20,20,20,20
6,47878,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Normal,BENIGN,-1,-1.200000e+07,-2.000000e+06,-1,-1.000000e+00,-1,0,252,119,20,20,20,20
7,61031,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Normal,BENIGN,-1,-1.930000e+08,-2.000000e+06,-1,-1.000000e+00,-1,0,0,8,20,20,20,20
8,61927,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Normal,BENIGN,-1,-1.200000e+07,-2.000000e+06,-1,-1.000000e+00,-1,0,63846,52560,20,20,20,20
9,70952,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Normal,BENIGN,114878507,1.488886e+03,1.801904e+00,-10,5.576627e+05,9999830,76,8192,8190,1472,1472,2684,20


## 12. Phân tích từng giá trị âm khác `-1`

Mục này tiếp tục từ `negative_examples`, nhưng tổng hợp toàn bộ dữ liệu thay vì chỉ xem mẫu. Với mỗi cột có giá trị âm hữu hạn khác `-1`, cell ghi nhận giá trị cụ thể, file nguồn, nhãn `Normal`/`Attack` và số dòng chứa giá trị đó.

Mục đích là phân biệt các giá trị lặp lại có thể là sentinel với các giá trị âm bất thường, đồng thời kiểm tra chúng có tập trung ở một file hoặc một nhãn hay không. Cell chỉ thống kê, không sửa hoặc loại bỏ dữ liệu.


In [35]:
negative_value_rows = []

for file in sorted(data_dir.glob("*.csv")):
    temp = pd.read_csv(file, encoding="latin-1")
    temp.columns = temp.columns.str.strip()
    numeric_part = temp.select_dtypes(include=np.number)
    binary_label = temp["Label"].eq("BENIGN").map({True: "Normal", False: "Attack"})
    finite_negative = numeric_part.lt(0) & np.isfinite(numeric_part)
    non_sentinel_negative = finite_negative & numeric_part.ne(-1)

    for column in numeric_part.columns:
        column_mask = non_sentinel_negative[column]
        if not column_mask.any():
            continue
        for value, value_count in numeric_part.loc[column_mask, column].value_counts().items():
            for label in ("Normal", "Attack"):
                rows = column_mask & binary_label.eq(label) & numeric_part[column].eq(value)
                count = int(rows.sum())
                if count:
                    negative_value_rows.append({
                        "Column": column,
                        "Value": value,
                        "File": file.name,
                        "BinaryLabel": label,
                        "Rows": count,
                    })
    del temp, numeric_part

negative_value_summary = pd.DataFrame(negative_value_rows)
negative_value_totals = (
    negative_value_summary
    .groupby(["Column", "Value", "BinaryLabel"], as_index=False)["Rows"]
    .sum()
    .sort_values(["Column", "Value", "BinaryLabel"])
)

print("Các giá trị âm hữu hạn khác -1 theo cột, giá trị và nhãn:")
display(negative_value_totals)

print("Các giá trị âm hữu hạn khác -1 theo cột và giá trị, gộp mọi file:")
display(
    negative_value_summary
    .groupby(["Column", "Value"], as_index=False)["Rows"]
    .sum()
    .sort_values(["Column", "Value"])
)


Các giá trị âm hữu hạn khác -1 theo cột, giá trị và nhãn:


,Column,Value,BinaryLabel,Rows
0,Bwd Header Length,-1.073741e+09,Normal,3
1,Bwd Header Length,-1.073741e+09,Normal,1
2,Bwd Header Length,-1.073741e+09,Normal,1
3,Bwd Header Length,-1.677705e+08,Normal,2
4,Bwd Header Length,-1.677705e+08,Normal,1
5,Bwd Header Length,-1.677704e+08,Normal,1
6,Bwd Header Length,-8.388522e+07,Normal,1
7,Bwd Header Length,-8.388518e+07,Normal,2
8,Bwd Header Length,-8.388514e+07,Normal,1
9,Bwd Header Length,-8.388514e+07,Normal,1


Các giá trị âm hữu hạn khác -1 theo cột và giá trị, gộp mọi file:


,Column,Value,Rows
0,Bwd Header Length,-1.073741e+09,3
1,Bwd Header Length,-1.073741e+09,1
2,Bwd Header Length,-1.073741e+09,1
3,Bwd Header Length,-1.677705e+08,2
4,Bwd Header Length,-1.677705e+08,1
5,Bwd Header Length,-1.677704e+08,1
6,Bwd Header Length,-8.388522e+07,1
7,Bwd Header Length,-8.388518e+07,2
8,Bwd Header Length,-8.388514e+07,1
9,Bwd Header Length,-8.388514e+07,1


## 13. Kiểm tra các dòng đại diện theo nhóm bất thường

Mục này lựa chọn một số dòng cho ba nhóm đã phát hiện: header length âm rất lớn, rate âm và `Flow IAT Min` âm trong các dòng Attack. Cell giữ file nguồn, chỉ số dòng, nhãn gốc và các feature liên quan để xem các bất thường có xuất hiện đồng thời hay không.

Cell chỉ mô tả dữ liệu; không sửa hoặc loại bỏ dòng.


In [37]:
anomaly_summary_rows = []
anomaly_example_frames = []

for file in sorted(data_dir.glob("*.csv")):
    temp = pd.read_csv(file, encoding="latin-1")
    temp.columns = temp.columns.str.strip()
    binary_label = temp["Label"].eq("BENIGN").map({True: "Normal", False: "Attack"})
    numeric_part = temp.select_dtypes(include=np.number)
    finite_negative = numeric_part.lt(0) & np.isfinite(numeric_part)

    header_columns = [c for c in ["Fwd Header Length", "Fwd Header Length.1", "Bwd Header Length"] if c in temp.columns]
    rate_columns = [c for c in ["Flow Bytes/s", "Flow Packets/s"] if c in temp.columns]
    header_mask = (numeric_part[header_columns].lt(-1) & np.isfinite(numeric_part[header_columns])).any(axis=1)
    rate_mask = (numeric_part[rate_columns].lt(-1) & np.isfinite(numeric_part[rate_columns])).any(axis=1)
    iat_attack_mask = temp["Flow IAT Min"].lt(-1) & binary_label.eq("Attack")

    group_masks = {
        "Header âm rất lớn": header_mask,
        "Rate âm": rate_mask,
        "Attack có Flow IAT Min âm": iat_attack_mask,
    }

    for group, mask in group_masks.items():
        for label in ("Normal", "Attack"):
            rows = mask & binary_label.eq(label)
            anomaly_summary_rows.append({"File": file.name, "IssueGroup": group, "BinaryLabel": label, "Rows": int(rows.sum())})

        if mask.any():
            selected_columns = ["Label", "Flow Duration", "Flow Bytes/s", "Flow Packets/s", "Flow IAT Min", "Flow IAT Mean", "Flow IAT Max", "Fwd Header Length", "Fwd Header Length.1", "Bwd Header Length", "min_seg_size_forward"]
            selected_columns = [c for c in selected_columns if c in temp.columns]
            example = temp.loc[mask, selected_columns].head(10).copy()
            example.insert(0, "IssueGroup", group)
            example.insert(0, "SourceFile", file.name)
            example.insert(2, "BinaryLabel", binary_label.loc[example.index].values)
            anomaly_example_frames.append(example.reset_index(names="SourceRow"))

    del temp, numeric_part

anomaly_summary = pd.DataFrame(anomaly_summary_rows)
anomaly_examples = pd.concat(anomaly_example_frames, ignore_index=True) if anomaly_example_frames else pd.DataFrame()

print("Số dòng theo nhóm bất thường, file và nhãn:")
display(anomaly_summary[anomaly_summary["Rows"] > 0])

print("Mẫu các dòng đại diện:")
display(anomaly_examples)


Số dòng theo nhóm bất thường, file và nhãn:


,File,IssueGroup,BinaryLabel,Rows
2,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Rate âm,Normal,2
5,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Attack có Flow IAT Min âm,Attack,1
8,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Rate âm,Normal,36
14,Friday-WorkingHours-Morning.pcap_ISCX.csv,Rate âm,Normal,9
18,Monday-WorkingHours.pcap_ISCX.csv,Header âm rất lớn,Normal,23
20,Monday-WorkingHours.pcap_ISCX.csv,Rate âm,Normal,15
26,Thursday-WorkingHours-Afternoon-Infilteration....,Rate âm,Normal,4
32,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,Rate âm,Normal,11
36,Tuesday-WorkingHours.pcap_ISCX.csv,Header âm rất lớn,Normal,11
38,Tuesday-WorkingHours.pcap_ISCX.csv,Rate âm,Normal,17


Mẫu các dòng đại diện:


,SourceRow,SourceFile,IssueGroup,BinaryLabel,Label,Flow Duration,Flow Bytes/s,Flow Packets/s,Flow IAT Min,Flow IAT Mean,Flow IAT Max,Fwd Header Length,Fwd Header Length.1,Bwd Header Length,min_seg_size_forward
0,200991,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Rate âm,Normal,BENIGN,-1,-1.200000e+07,-2.000000e+06,-1,-1.000000e+00,-1,20,20,20,20
1,224100,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Rate âm,Normal,BENIGN,-1,-1.200000e+07,-2.000000e+06,-1,-1.000000e+00,-1,20,20,20,20
2,96829,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Attack có Flow IAT Min âm,Attack,DDoS,20933,5.557254e+05,3.344002e+02,-12,3.488833e+03,20325,104,104,60,20
3,47878,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Rate âm,Normal,BENIGN,-1,-1.200000e+07,-2.000000e+06,-1,-1.000000e+00,-1,20,20,20,20
4,61031,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Rate âm,Normal,BENIGN,-1,-1.930000e+08,-2.000000e+06,-1,-1.000000e+00,-1,20,20,20,20
5,61927,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Rate âm,Normal,BENIGN,-1,-1.200000e+07,-2.000000e+06,-1,-1.000000e+00,-1,20,20,20,20
6,86737,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Rate âm,Normal,BENIGN,-1,-1.200000e+07,-2.000000e+06,-1,-1.000000e+00,-1,20,20,20,20
7,90372,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Rate âm,Normal,BENIGN,-1,-8.000000e+06,-2.000000e+06,-1,-1.000000e+00,-1,20,20,24,20
8,90965,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Rate âm,Normal,BENIGN,-1,-8.000000e+06,-2.000000e+06,-1,-1.000000e+00,-1,20,20,24,20
9,98622,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Rate âm,Normal,BENIGN,-1,-8.000000e+06,-2.000000e+06,-1,-1.000000e+00,-1,20,20,24,20


## 14. Kiểm tra chồng lấp với sentinel `-1`

Mục này kiểm tra các dòng có rate hoặc header length âm có đồng thời mang các sentinel `-1` hay không. Cell cũng tách riêng các dòng Attack có `Flow IAT Min` âm nhưng các feature liên quan khác vẫn hợp lệ.

Mục tiêu là xác định nên xử lý theo cụm sentinel, theo feature, hay theo từng dòng. Cell chỉ thống kê, không thay đổi dữ liệu.


In [39]:
overlap_rows = []
attack_iat_rows = []

for file in sorted(data_dir.glob("*.csv")):
    temp = pd.read_csv(file, encoding="latin-1")
    temp.columns = temp.columns.str.strip()
    binary_label = temp["Label"].eq("BENIGN").map({True: "Normal", False: "Attack"})
    numeric_part = temp.select_dtypes(include=np.number)
    finite_negative = numeric_part.lt(0) & np.isfinite(numeric_part)

    header_columns = [c for c in ["Fwd Header Length", "Fwd Header Length.1", "Bwd Header Length"] if c in temp.columns]
    rate_columns = [c for c in ["Flow Bytes/s", "Flow Packets/s"] if c in temp.columns]
    header_mask = (numeric_part[header_columns].lt(-1) & np.isfinite(numeric_part[header_columns])).any(axis=1)
    rate_mask = (numeric_part[rate_columns].lt(-1) & np.isfinite(numeric_part[rate_columns])).any(axis=1)
    iat_attack_mask = temp["Flow IAT Min"].lt(-1) & binary_label.eq("Attack")

    for group, mask in [("Header âm rất lớn", header_mask), ("Rate âm", rate_mask)]:
        for label in ("Normal", "Attack"):
            rows = mask & binary_label.eq(label)
            overlap_rows.append({
                "File": file.name,
                "IssueGroup": group,
                "BinaryLabel": label,
                "Rows": int(rows.sum()),
                "Flow Duration = -1": int((rows & temp["Flow Duration"].eq(-1)).sum()),
                "Flow IAT Min = -1": int((rows & temp["Flow IAT Min"].eq(-1)).sum()),
                "Flow IAT Mean = -1": int((rows & temp["Flow IAT Mean"].eq(-1)).sum()),
                "Flow IAT Max = -1": int((rows & temp["Flow IAT Max"].eq(-1)).sum()),
            })

    if iat_attack_mask.any():
        rows = iat_attack_mask
        attack_iat_rows.append({
            "File": file.name,
            "Attack rows with Flow IAT Min negative": int(rows.sum()),
            "Flow Duration positive": int((rows & temp["Flow Duration"].gt(0)).sum()),
            "Flow Bytes/s positive": int((rows & temp["Flow Bytes/s"].gt(0)).sum()),
            "Flow Packets/s positive": int((rows & temp["Flow Packets/s"].gt(0)).sum()),
            "Header values non-negative": int((rows & (numeric_part[header_columns].ge(0).all(axis=1))).sum()),
        })
    del temp, numeric_part

overlap_summary = pd.DataFrame(overlap_rows)
attack_iat_summary = pd.DataFrame(attack_iat_rows)

print("Mức chồng lấp giữa rate/header âm và sentinel -1:")
display(overlap_summary[overlap_summary["Rows"] > 0])

print("Các dòng Attack có Flow IAT Min âm:")
display(attack_iat_summary)


Mức chồng lấp giữa rate/header âm và sentinel -1:


,File,IssueGroup,BinaryLabel,Rows,Flow Duration = -1,Flow IAT Min = -1,Flow IAT Mean = -1,Flow IAT Max = -1
2,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Rate âm,Normal,2,2,2,2,2
6,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Rate âm,Normal,36,33,33,33,33
10,Friday-WorkingHours-Morning.pcap_ISCX.csv,Rate âm,Normal,9,8,8,8,8
12,Monday-WorkingHours.pcap_ISCX.csv,Header âm rất lớn,Normal,23,0,0,0,0
14,Monday-WorkingHours.pcap_ISCX.csv,Rate âm,Normal,15,15,15,15,15
18,Thursday-WorkingHours-Afternoon-Infilteration....,Rate âm,Normal,4,3,3,3,3
22,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,Rate âm,Normal,11,11,11,11,11
24,Tuesday-WorkingHours.pcap_ISCX.csv,Header âm rất lớn,Normal,11,0,0,0,0
26,Tuesday-WorkingHours.pcap_ISCX.csv,Rate âm,Normal,17,16,16,16,16
28,Wednesday-workingHours.pcap_ISCX.csv,Header âm rất lớn,Normal,1,0,0,0,0


Các dòng Attack có Flow IAT Min âm:


,File,Attack rows with Flow IAT Min negative,Flow Duration positive,Flow Bytes/s positive,Flow Packets/s positive,Header values non-negative
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,1,1,1,1,1
1,Wednesday-workingHours.pcap_ISCX.csv,14,14,13,14,14


## 15. Kết luận Data Exploration và định hướng Data Preprocessing

Qua quá trình khảo sát 8 file CSV của CICIDS2017, các vấn đề chất lượng dữ liệu đã được xác định như sau:

### 15.1. Các kết quả chính

1. **NaN và Infinity**
   - Có 1.358 ô NaN và 4.376 ô Infinity.
   - Có 2.867 dòng chứa ít nhất một NaN hoặc Infinity; tất cả đều có `Flow Duration = 0`.
   - Chưa thực hiện xóa hoặc thay thế dữ liệu.

2. **Giá trị âm hữu hạn**
   - `Init_Win_bytes_forward` và `Init_Win_bytes_backward` có số lượng lớn giá trị `-1`, nhiều khả năng là giá trị quy ước (sentinel), cần xử lý theo ý nghĩa từng feature.
   - Có 115 dòng Normal chứa rate âm; 109 dòng có `Flow Duration = -1`.
   - Có 35 dòng Normal chứa header length âm rất lớn, không chồng lấp với các sentinel thời gian `-1` đã kiểm tra.
   - Có 194 dòng Attack chứa `Flow IAT Min` âm; trong đó 15 dòng có giá trị nhỏ hơn `-1`. Cả 15 dòng này có `Flow Duration > 0` và header không âm.
   - Chưa có đủ căn cứ để tự động loại bỏ tất cả các dòng có giá trị âm.

3. **Dữ liệu trùng lặp**
   - Phát hiện 256.479 dòng trùng hoàn toàn trong nội bộ các file CSV.
   - Chưa kiểm tra trùng lặp giữa các file khác nhau và chưa thực hiện loại bỏ duplicate.
   - Cần xem xét ảnh hưởng của duplicate đến chia Train/Test, vì Random Split có thể cho kết quả lạc quan nếu bản ghi trùng xuất hiện ở cả hai tập.

### 15.2. Nguyên tắc tiền xử lý dự kiến

1. Không chỉnh sửa trực tiếp các file CSV gốc.
2. Giữ `SourceFile`, `CaptureDay`, `OriginalLabel`, `BinaryLabel` ngoài ma trận feature.
3. Xử lý NaN và Infinity theo quy tắc phù hợp; cân nhắc loại dòng không hợp lệ hoặc thay thế giá trị thiếu, không mặc định imputation cho mọi trường hợp.
4. Không mặc định tất cả giá trị âm là lỗi; phân biệt sentinel `-1`, giá trị bất thường và trường hợp cần điều tra thêm.
5. Không tự động xóa các dòng Attack có `Flow IAT Min` âm khi chưa xác định nguyên nhân và ảnh hưởng.
6. Kiểm tra và xử lý duplicate phù hợp, tránh để bản ghi trùng xuất hiện đồng thời trong Train và Test.
7. **Ngăn ngừa Data Leakage:** tách Train/Test trước các bước cần học tham số; chỉ fit imputer, scaler, feature selector và SMOTE (nếu sử dụng) trên Train; không dùng Test để lựa chọn hoặc tinh chỉnh mô hình.
8. Ghi nhận số lượng mẫu Normal/Attack trước và sau mỗi bước cleaning.

### 15.3. Định hướng các giai đoạn tiếp theo

Data Exploration đã đủ cơ sở để chuyển sang thiết kế quy trình Data Preprocessing. Các quyết định cleaning cụ thể sẽ được thử nghiệm và xác minh trong notebook preprocessing, không thực hiện trực tiếp tại notebook này.

Dự kiến áp dụng Random Split, Day-Disjoint Split và LOFO nếu còn thời gian. Đánh giá bằng Accuracy, Precision, Recall, F1-score, Macro-F1, FPR, FNR và Confusion Matrix. Mục tiêu vẫn là NIDS phân loại nhị phân Normal/Attack.
